In [1]:
import pandas as pd
import numpy as np
import os

print("Starting Data Cleaning Process...\n")

# Ensure processed directory exists
os.makedirs('../data/processed', exist_ok=True)

# 1. Load Raw Datasets
orders = pd.read_csv('../data/olist_orders_dataset.csv')
items = pd.read_csv('../data/olist_order_items_dataset.csv')
products = pd.read_csv('../data/olist_products_dataset.csv')
payments = pd.read_csv('../data/olist_order_payments_dataset.csv')
reviews = pd.read_csv('../data/olist_order_reviews_dataset.csv')
customers = pd.read_csv('../data/olist_customers_dataset.csv')
sellers = pd.read_csv('../data/olist_sellers_dataset.csv')
translations = pd.read_csv('../data/product_category_name_translation.csv')

# --- CLEANING STEP 1: Parse Timestamps ---
print("1. Parsing date & timestamp columns...")
date_cols = [
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

# --- CLEANING STEP 2: Portuguese to English Product Category Mapping ---
print("2. Mapping product categories to English...")
products = products.merge(translations, on='product_category_name', how='left')
# Fill missing translations with 'unknown'
products['product_category_name_english'] = products['product_category_name_english'].fillna('unknown')
# Drop original Portuguese column to avoid duplicate naming downstream
products.drop(columns=['product_category_name'], inplace=True)

# --- CLEANING STEP 3: Handle Delivery Delays & Feature Engineering ---
print("3. Engineering delivery metrics...")
# Delivery delay in days (Negative means delivered earlier than estimated)
orders['delivery_delay_days'] = (
    orders['order_delivered_customer_date'] - orders['order_estimated_delivery_date']
).dt.total_seconds() / (24 * 3600)

orders['is_delayed'] = orders['delivery_delay_days'] > 0

# Total actual delivery time in days
orders['delivery_time_days'] = (
    orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']
).dt.total_seconds() / (24 * 3600)

# Extract time dimensions for trend analysis
orders['purchase_year'] = orders['order_purchase_timestamp'].dt.year
orders['purchase_month'] = orders['order_purchase_timestamp'].dt.to_period('M').astype(str)
orders['purchase_dayofweek'] = orders['order_purchase_timestamp'].dt.day_name()

# --- CLEANING STEP 4: Fill Missing Review Comments ---
print("4. Cleaning review scores & feedback text...")
reviews['review_comment_title'] = reviews['review_comment_title'].fillna('No Title')
reviews['review_comment_message'] = reviews['review_comment_message'].fillna('No Comment Provided')

# --- CLEANING STEP 5: Export Processed Datasets ---
print("\n5. Exporting clean datasets to data/processed/...")
orders.to_csv('../data/processed/clean_orders.csv', index=False)
items.to_csv('../data/processed/clean_order_items.csv', index=False)
products.to_csv('../data/processed/clean_products.csv', index=False)
payments.to_csv('../data/processed/clean_payments.csv', index=False)
reviews.to_csv('../data/processed/clean_reviews.csv', index=False)
customers.to_csv('../data/processed/clean_customers.csv', index=False)
sellers.to_csv('../data/processed/clean_sellers.csv', index=False)

print("\nSUCCESS! All clean datasets exported successfully to data/processed/")

Starting Data Cleaning Process...

1. Parsing date & timestamp columns...
2. Mapping product categories to English...
3. Engineering delivery metrics...
4. Cleaning review scores & feedback text...

5. Exporting clean datasets to data/processed/...

SUCCESS! All clean datasets exported successfully to data/processed/
